# Model Comparison

Коротко: сравниваем несколько моделей на одинаковом train split и выбираем кандидатов для тюнинга.


## Setup / Настройка


In [1]:
# База / core
from pathlib import Path
import time

import pandas as pd
import seaborn as sns
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier

# Sklearn / ML
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import ExtraTreesClassifier, HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Корень проекта / project root
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "requirements.txt").exists())

# Стиль / style
sns.set_theme(style="whitegrid", palette="Set2")
pd.set_option("display.max_columns", 80)
pd.set_option("display.max_colwidth", 70)


## Data / Данные


In [2]:
# Данные после FE / data after feature engineering
df_model = pd.read_parquet(ROOT / "data" / "titanic_fe.parquet")

X = df_model.drop(columns="survived")
y = df_model["survived"].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)


## Features / Признаки


In [3]:
# признаки для предобработки / features for preprocessing
numeric_features = (
    X_train
    .select_dtypes(include="number")
    .columns
    .tolist()
)

categorical_features = (
    X_train
    .select_dtypes(include=["object", "category", "string"])
    .columns
    .tolist()
)

## Preprocessing / Предобработка


In [4]:
# Preprocessing for tree models
tree_preprocessor = ColumnTransformer([
    ("num", SimpleImputer(strategy="median"), numeric_features),
    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
    ]), categorical_features)
])

In [5]:
logreg_preprocessor = ColumnTransformer([
    ("num", Pipeline([("imputer", SimpleImputer(strategy="median")),
                      ("scaler", StandardScaler())]), numeric_features),
    ("cat", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")),
                      ("encoder", OneHotEncoder(handle_unknown="ignore"))]), categorical_features)
])

## Models / Модели


In [6]:
# Модели / models
models = {
    "LogisticRegression": Pipeline([
        ("preprocessor", logreg_preprocessor),
        ("model", LogisticRegression(max_iter=5000)),
    ]),
    "RandomForest": Pipeline([
        ("preprocessor", tree_preprocessor),
        ("model", RandomForestClassifier(
            n_estimators=1000,
            max_depth=8,
            min_samples_leaf=3,
            min_samples_split=10,
            max_features="sqrt",
            random_state=42,
            n_jobs=-1,
        )),
    ]),
    "ExtraTrees": Pipeline([
        ("preprocessor", tree_preprocessor),
        ("model", ExtraTreesClassifier(random_state=42, n_jobs=-1)),
    ]),
    "HistGradientBoosting": Pipeline([
        ("preprocessor", tree_preprocessor),
        ("model", HistGradientBoostingClassifier(random_state=42)),
    ]),
    "CatBoost": Pipeline([
        ("preprocessor", tree_preprocessor),
        ("model", CatBoostClassifier(verbose=0, random_state=42)),
    ]),
    "LightGBM": Pipeline([
        ("preprocessor", tree_preprocessor),
        ("model", LGBMClassifier(random_state=42, verbose=-1)),
    ]),
    "XGBoost": Pipeline([
        ("preprocessor", tree_preprocessor),
        ("model", XGBClassifier(random_state=42)),
    ]),
}


## Cross-Validation / Кросс-валидация


In [7]:
# CV-сравнение / CV comparison
results = []

for name, model in models.items():
    print("\n" + "=" * 50)
    print(name)
    start = time.time()

    try:
        scores = cross_val_score(
            model,
            X_train,
            y_train,
            cv=5,
            scoring="accuracy",
            n_jobs=-1,
            error_score="raise",
        )
        elapsed = time.time() - start
        print(f"CV: {scores.mean():.4f} ± {scores.std():.4f}")
        print(f"Time: {elapsed:.1f} sec")

        results.append({
            "model": name,
            "cv_mean": scores.mean(),
            "cv_std": scores.std(),
            "time_sec": elapsed,
        })
    except Exception as e:
        print("ERROR:")
        print(e)

results = pd.DataFrame(results).sort_values(by="cv_mean", ascending=False)
display(results.round(4))



LogisticRegression


CV: 0.8051 ± 0.0179
Time: 1.8 sec

RandomForest
CV: 0.8137 ± 0.0229
Time: 4.6 sec

ExtraTrees
CV: 0.7727 ± 0.0233
Time: 1.5 sec

HistGradientBoosting
CV: 0.7994 ± 0.0246
Time: 1.3 sec

CatBoost
CV: 0.7994 ± 0.0214
Time: 5.0 sec

LightGBM


/home/mchunikhin/miniconda3/lib/python3.13/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/mchunikhin/miniconda3/lib/python3.13/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/mchunikhin/miniconda3/lib/python3.13/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/mchunikhin/miniconda3/lib/python3.13/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/mchunikhin/miniconda3/lib/python3.13/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted w

CV: 0.7994 ± 0.0305
Time: 1.8 sec

XGBoost
CV: 0.7803 ± 0.0351
Time: 0.7 sec


,model,cv_mean,cv_std,time_sec
1,RandomForest,0.8137,0.0229,4.5744
0,LogisticRegression,0.8051,0.0179,1.8161
3,HistGradientBoosting,0.7994,0.0246,1.3117
4,CatBoost,0.7994,0.0214,4.9828
5,LightGBM,0.7994,0.0305,1.7981
6,XGBoost,0.7803,0.0351,0.6584
2,ExtraTrees,0.7727,0.0233,1.5140


## Входы, выходы и выводы / Inputs, outputs & conclusions

- Вход: `data/titanic_fe.parquet` из ноутбука 02.
- Выход: таблица CV-качества и времени для нескольких моделей.
- Вывод: лучшие модели из сравнения стоит передать в тюнинг; сравнение честное, потому что train/test split и preprocessing одинаковые внутри `Pipeline`.
